In [1]:
import os
import commot as ct
import scanpy as sc
import squidpy as sq
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap 
from numpy.random import default_rng
from anndata import AnnData
sc.logging.print_header() 
print(f"squidpy=={sq.__version__}") 

squidpy==1.6.5


In [2]:
# Create anndata from 10x output 

path = os.path.expanduser("C:/01. Research/01. mIDH (hypoxia)/02. Analysis/03. Analysis (stRNA-seq)/06. CCC/BWH24/")
h5_path = os.path.join(path, "filtered_feature_bc_matrix.h5")
print(os.path.exists(h5_path)) 

adata = sq.read.visium(path = path, 
                       counts_file = 'filtered_feature_bc_matrix.h5', 
                       load_images = True) 

True


In [3]:
# Fixing the non-unique gene names 

adata.var_names_make_unique() 

In [4]:
# Basic preprocessing 

sc.pp.normalize_total(adata, inplace = True)
sc.pp.log1p(adata)

In [5]:
# Specify ligand-receptor pairs 

df_ligrec = pd.DataFrame([['EGFR', 'HBEGF', 'EGF_pathway']], columns = ['ligand', 'receptor', 'pathway']) 
df_ligrec['ligand'] = df_ligrec['ligand'].str.upper()
df_ligrec['receptor'] = df_ligrec['receptor'].str.upper()

adata.var_names = adata.var_names.str.upper()

In [6]:
# Constructing cell-cell communication networks with a spatial distance constraint of 250µm 

adata.var_names_make_unique() 
print(adata.var_names.is_unique) 
ct.tl.spatial_communication(adata, 
                            database_name = 'user_database', 
                            df_ligrec = df_ligrec, 
                            dis_thr = 250, 
                            heteromeric = True, 
                            pathway_sum = True) 

True


In [7]:
print(list(adata.obsm.keys())) 
print(adata.obsm['commot-user_database-sum-sender'].columns)
print(adata.obsm['commot-user_database-sum-receiver'].columns)

['spatial', 'commot-user_database-sum-sender', 'commot-user_database-sum-receiver']
Index(['s-EGFR-HBEGF', 's-total-total', 's-EGF_pathway'], dtype='object')
Index(['r-EGFR-HBEGF', 'r-total-total', 'r-EGF_pathway'], dtype='object')


In [8]:
# Define output directory

output_dir = os.path.expanduser("C:/01. Research/01. mIDH (hypoxia)/02. Analysis/03. Analysis (stRNA-seq)/06. CCC/plots")
os.makedirs(output_dir, exist_ok = True) 

In [9]:
print(list(adata.obsm.keys())) 
print(adata.obsm['commot-user_database-sum-sender'].columns)
print(adata.obsm['commot-user_database-sum-receiver'].columns)

['spatial', 'commot-user_database-sum-sender', 'commot-user_database-sum-receiver']
Index(['s-EGFR-HBEGF', 's-total-total', 's-EGF_pathway'], dtype='object')
Index(['r-EGFR-HBEGF', 'r-total-total', 'r-EGF_pathway'], dtype='object')


In [10]:
ct.tl.communication_direction(adata, database_name = 'user_database', lr_pair = ('EGFR','HBEGF'), k=5) 

In [11]:
print([k for k in adata.obsm.keys() if 'sender_vf' in k])

['commot_sender_vf-user_database-EGFR-HBEGF']


In [12]:
adata.obsm['commot-user_database-sum-sender']
adata.obsm['commot-user_database-sum-receiver'] 

,r-EGFR-HBEGF,r-total-total,r-EGF_pathway
AAACAAGTATCTCCCA-1,0.000000,0.000000,0.000000
AAACAATCTACTAGCA-1,0.000000,0.000000,0.000000
AAACAGAGCGACTCCT-1,0.000000,0.000000,0.000000
AAACAGTGTTCCTGGG-1,0.000000,0.000000,0.000000
AAACATTTCCCGGATT-1,0.000000,0.000000,0.000000
...,...,...,...
TTGTTCAGTGTGCTAC-1,0.000000,0.000000,0.000000
TTGTTGTGTGTCAAGA-1,0.565875,0.565875,0.565875
TTGTTTCACATCCAGG-1,0.000000,0.000000,0.000000
TTGTTTGTATTACACG-1,0.000000,0.000000,0.000000


In [13]:
# Plot sender and receiver togather 

plt.ioff() 

pts = adata.obsm['spatial']
s = adata.obsm['commot-user_database-sum-sender']['s-EGFR-HBEGF']
r = adata.obsm['commot-user_database-sum-receiver']['r-EGFR-HBEGF']

fig, ax = plt.subplots(figsize=(6, 6)) 

sc1 = ax.scatter(pts[:, 0], pts[:, 1], c = s, cmap = 'Blues', marker = 's', s = 20, alpha = 0.8, label = 'Sender', zorder = 2)
sc2 = ax.scatter(pts[:, 0], pts[:, 1], c = r, cmap = 'Reds', marker = '^', s = 20, alpha = 0.8, label = 'Receiver', zorder = 3)
ax.set_title('HBEGF_sender/EGFR_receiver')
ax.legend() 

cbar1 = plt.colorbar(sc1, ax = ax, fraction = 0.046, pad = 0.04)
cbar1.set_label('Sender score')
cbar2 = plt.colorbar(sc2, ax = ax, fraction = 0.046, pad = 0.1)
cbar2.set_label('Receiver score')

pdf_sr = os.path.join(output_dir, "01. Sender_receiver_shapes.pdf")
fig.savefig(pdf_sr, format = 'pdf', dpi = 300, bbox_inches = 'tight')
plt.close(fig)
print("Saved PDF:", pdf_sr)

Saved PDF: C:/01. Research/01. mIDH (hypoxia)/02. Analysis/03. Analysis (stRNA-seq)/06. CCC/plots\01. Sender_receiver_shapes.pdf


In [14]:
# Custom color palette 

colors_s = ['#d7e5b7', '#78a516', '#446300'] 
custom_cmap_s = LinearSegmentedColormap.from_list('my_cmap', colors_s) 
colors_r = ['#f7d0e2', '#C95792', '#7C4585'] 
custom_cmap_r = LinearSegmentedColormap.from_list('my_cmap', colors_r) 

In [15]:
# Visualize signaling directions in a grid 

plt.ioff()

ct.pl.plot_cell_communication(adata, 
                              database_name = 'user_database', 
                              lr_pair = ('EGFR','HBEGF'), 
                              plot_method = 'grid', 
                              background_legend = True, 
                              scale = 0.00001, 
                              ndsize = 30, 
                              grid_density = 0.4, 
                              summary = 'sender', 
                              background = 'summary', 
                              clustering = 'leiden', 
                              cmap = custom_cmap_s, 
                              normalize_v = True, 
                              normalize_v_quantile = 0.995, 
                             filename = os.path.join(output_dir, "02. BWH24_sender_grid.pdf")) 
plt.close('all') 

ct.pl.plot_cell_communication(adata, 
                              database_name = 'user_database', 
                              lr_pair = ('EGFR','HBEGF'), 
                              plot_method = 'grid', 
                              background_legend = True, 
                              scale = 0.00001, 
                              ndsize = 30, 
                              grid_density = 0.4, 
                              summary = 'receiver', 
                              background = 'summary', 
                              clustering = 'leiden', 
                              cmap = custom_cmap_r, 
                              normalize_v = True, 
                              normalize_v_quantile = 0.995, 
                             filename = os.path.join(output_dir, "02. BWH24_receiver_grid.pdf")) 

plt.close('all')
print("Saved grid plots")

Saved grid plots


In [16]:
# Visualize signaling directions in a stream 

plt.ioff()

ct.pl.plot_cell_communication(adata, 
                              database_name = 'user_database', 
                              lr_pair = ('EGFR','HBEGF'), 
                              plot_method = 'stream', 
                              background_legend = True, 
                              scale = 0.00001, 
                              ndsize = 50, 
                              grid_density = 0.4, 
                              summary = 'sender', 
                              background = 'summary', 
                              clustering = 'leiden', 
                              cmap = custom_cmap_s, 
                              arrow_color = 'black', 
                              stream_density = 1.0, 
                              stream_linewidth = 2, 
                              normalize_v = False, 
                              normalize_v_quantile = 0.995)

plt.savefig(os.path.join(output_dir, "03. BWH24_sender_stream.png"), dpi = 300, bbox_inches = "tight")
plt.close()

ct.pl.plot_cell_communication(adata, 
                              database_name = 'user_database', 
                              lr_pair = ('EGFR','HBEGF'), 
                              plot_method = 'stream', 
                              background_legend = True, 
                              scale = 0.00001, 
                              ndsize = 50, 
                              grid_density = 0.4, 
                              summary = 'receiver', 
                              background = 'summary', 
                              clustering = 'leiden', 
                              cmap = custom_cmap_r, 
                              arrow_color = 'black', 
                              stream_density = 1.0, 
                              stream_linewidth = 2, 
                              normalize_v = False, 
                              normalize_v_quantile = 0.995)

plt.savefig(os.path.join(output_dir, "03. BWH24_receiver_stream.png"), dpi = 300, bbox_inches = "tight")
plt.close()
print("Saved stream plots.") 

Saved stream plots.
